# Imports and Paths

In [ ]:
from TELF.applications import Termite
from TELF.applications.Termite.neo4j_termite import ENTITY, RETURN_TYPE, ATTRIBUTES, ET,\
                              YEAR_TYPE, FROM_COL, DOCUMENT_TYPE,\
                              ROW_INDEX, ATTR_COL, ATTR_NAME, TT,\
                              R, DOCUMENT_YEAR_RELATION, HT,\
                              AUTHOR_DOCUMENT_RELATION, AUTHOR_ID_TYPE,\
                              EXTRACT_H, DOCUMENT_CITES_RELATION,\
                              DOCUMENT_CITED_RELATION, DOCUMENT_TYPE_SCOPUS,\
                              EXTRACT_T, PAIRING, HEAD_TO_MANY
from copy import deepcopy
import pandas as pd

username, password  = "neo4j", "local_password"
token = None
URI = "neo4j://localhost:7666"
credentials = (URI, (username, password))


termite = Termite(kg_credentials=credentials,  
                  vector_uri="http://localhost:19530",
                  db_nme="default",
                  token=token,
                  verbose=False)

### Define Paths for source and destination data

In [1]:
raw_csv_path = "../../data/sample2.csv"
triplets_path = "./01_termite_output/sample_triplets_data.csv"
import os
os.makedirs('01_termite_output', exist_ok=True)

# NEO4J INJECTION

### Define Extraction Functions for each of the relations with complex data

In [ ]:
def list_split_no_attrs(data, split_with=';'):
    returnable_entities = []
    if type(data) == str:
        split_values = data.split(split_with)
        for entity_value in split_values:
            entity_returnable = deepcopy(RETURN_TYPE)
            entity_returnable[ENTITY] = entity_value
            returnable_entities.append(entity_returnable)
        return returnable_entities
    else:
        return [deepcopy(RETURN_TYPE)]

def get_cites(args):
    data_string = args['data']
    return list_split_no_attrs(data_string.citations)
     
def get_cited(args):
    data_string = args['data']
    return list_split_no_attrs(data_string.references)
    
def get_authors_ID(args):
    data_string = args['data']

    data = data_string.s2_author_ids
    if type(data) == str:
        split_author_ids= data.split(';')
        authors = data_string.s2_authors
        if type(authors) == str:
                authors_split_values = authors.split(';')
        returnable_entities = []
    
        for entity_value, attribute in zip(split_author_ids, authors_split_values ):
            entity_returnable = deepcopy(RETURN_TYPE)
            entity_returnable[ENTITY] = entity_value
            entity_returnable[ATTRIBUTES] = [('name', attribute)]
            returnable_entities.append(entity_returnable)
        return returnable_entities
    else:
        return [deepcopy(RETURN_TYPE)]
    

### Define the mapping from the columns to the desired structure

In [ ]:
"""
Entities need to tell the type, the column it comes from, any associates attributes and how to get those attribuetes if embedded in a data structure inline.
Relations need to say the head type, relation type, tail type, and how to get the tail if the tail is compossed of multiple entities embedded inline. 
The fourth passed item is the tail entity retrieval, and should return a list of dictionaries, where each dictionary contains: 'entity','weight','attributes'
"""
column_triplet_map =  {
    'ENTITIES':
    [
        {ET:YEAR_TYPE, FROM_COL: 'year'},
        {ET:DOCUMENT_TYPE, FROM_COL: ROW_INDEX, ATTR_COL:[{FROM_COL: 'title', ATTR_NAME:'Title', },
                                                          {FROM_COL: 's2id', ATTR_NAME:'S2ID', },
                                                          {FROM_COL: 'doi', ATTR_NAME:'DOI', },
                                                          ]},
    ],
    'RELATIONS':
    [   
        {HT:DOCUMENT_TYPE, R:DOCUMENT_YEAR_RELATION, TT:YEAR_TYPE},
        {HT:AUTHOR_ID_TYPE, R:AUTHOR_DOCUMENT_RELATION, TT:DOCUMENT_TYPE,  EXTRACT_H: get_authors_ID},
        {HT:DOCUMENT_TYPE, R:DOCUMENT_CITES_RELATION, TT:DOCUMENT_TYPE_SCOPUS,  EXTRACT_T: get_cites, PAIRING: HEAD_TO_MANY},
        {HT:DOCUMENT_TYPE, R:DOCUMENT_CITED_RELATION, TT:DOCUMENT_TYPE_SCOPUS,  EXTRACT_T: get_cited, PAIRING: HEAD_TO_MANY},
    ]
}


### Construct Triplets from Raw Data

In [ ]:
termite.from_csv_to_triplets(raw_csv_path, triplets_path, column_triplet_map)

### Inject Triplets into graph

In [ ]:
termite.update_database_multithreaded(triplets_path)


# MILVUS INJECTION


In [2]:
# test_opensearch_termite.py
import os
import pandas as pd

from TELF.applications import Termite

# --- ensure backend is OpenSearch (this is the default anyway) ---
os.environ["EMBEDDING_STORE"] = "opensearch"
os.environ["OS_HOST"] = "localhost"
os.environ["OS_PORT"] = "9200"
os.environ["OS_USE_SSL"] = "false"

# --- init Termite (Neo4j creds not required for this test) ---
t = Termite(
    kg_credentials=None,
    verbose=True,
    # you can swap the model if you want; default is "malteos/scincl"
    model_name="malteos/scincl",
)

# --- load a small dataframe ---
df = pd.read_csv(raw_csv_path)

# --- compute embeddings (returns dict: index -> vector) ---
emb_map = t.compute_embeddings(df, model_name="malteos/scincl")
# convert to row-ordered list to align with df rows
embeddings = [emb_map[i] for i in df.index.tolist()]

# infer embedding dimension from the first row
dim = len(embeddings[0])

index_name = "termite_vectors_test"

# --- create the OpenSearch k-NN index (HNSW, cosine by default) ---
t.make_vector_schema(collection_name=index_name, dim=dim, metric="cosine")

# --- shape data & upsert ---
data = t.df_to_data(
    embeddings=embeddings,
    df_path=raw_csv_path,
    columns_collection_map={
        # map CSV columns into the payload keys expected by the store
        "id": "eid",
        "text": "abstract",
    },
)
t.inject_vectors(index_name, data)

# --- quick search: use the first vector as the query ---
hits = t.search_vectors(index_name, embeddings[0], k=3)
print("\nTop-3 hits:")
for _id, score, payload in hits:
    print(f"  score={score:.4f}  id={_id}  text={payload.get('text')}")


/Users/barron/anaconda3/envs/TELF/lib/python3.11/site-packages/pymilvus/client/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution



Top-3 hits:
  score=1.0000  id=0  text=None
  score=0.9441  id=213  text=None
  score=0.9423  id=58  text=None


In [8]:
# test_opensearch_termite_complete.py
"""
End-to-end OpenSearch + TELF Termite example (OpenSearch 2.13)

- Uses Termite to compute embeddings (malteos/scincl, 768-D).
- Creates a knn_vector index (HNSW, cosine).
- Upserts docs with id + embedding + text (top-level "_source.text").
- Queries with the OpenSearch-native k-NN body and prints text.

Requirements:
  pip install pandas opensearch-py TELF

Notes:
  - Set raw_csv_path to your CSV.
  - CSV should have an 'eid' column (stringable doc id) and an 'abstract' column (text).
  - If your column names differ, change ID_COL/TEXT_COL below.
"""

import os
import warnings
import pandas as pd
from typing import List, Dict

# Suppress the pymilvus pkg_resources deprecation warning that TELF pulls in
warnings.filterwarnings("ignore", category=UserWarning, message="pkg_resources is deprecated")

from TELF.applications import Termite
from opensearchpy import OpenSearch, helpers

# ---------- USER CONFIG ----------
INDEX_NAME   = "termite_vectors_test"
ID_COL       = "eid"                # <-- change if your id column has a different name
TEXT_COL     = "abstract"           # <-- change if your text column has a different name
MODEL_NAME   = "malteos/scincl"     # 768-D sentence embeddings
RECREATE_INDEX = True               # set False to append
TOP_K        = 3
# ---------------------------------

# OpenSearch connection (dev defaults)
OS_HOST = os.environ.get("OS_HOST", "localhost")
OS_PORT = int(os.environ.get("OS_PORT", "9200"))
OS_USE_SSL = os.environ.get("OS_USE_SSL", "false").lower() == "true"

def get_client() -> OpenSearch:
    return OpenSearch(
        hosts=[{"host": OS_HOST, "port": OS_PORT}],
        use_ssl=OS_USE_SSL,
        verify_certs=False,
        http_compress=True,
        timeout=60,
    )

def ensure_index(client: OpenSearch, index: str, dim: int, recreate: bool = False) -> None:
    if recreate and client.indices.exists(index=index):
        client.indices.delete(index=index, ignore=[404])

    if client.indices.exists(index=index):
        return

    body = {
        "settings": {
            "index": {
                "knn": True,
                "number_of_shards": 1,
                "number_of_replicas": 0,
                # Optional: slightly higher ef_search for better recall (set at index level on OS 2.13)
                "knn.algo_param.ef_search": 128,
            }
        },
        "mappings": {
            "properties": {
                "id":   {"type": "keyword"},
                "text": {"type": "text"},
                "metadata": {"type": "object"},
                "embedding": {
                    "type": "knn_vector",
                    "dimension": int(dim),
                    "method": {
                        "name": "hnsw",
                        "engine": "nmslib",
                        "space_type": "cosinesimil"
                    }
                }
            }
        }
    }
    client.indices.create(index=index, body=body)

def to_float_list(vec) -> List[float]:
    # Normalize numpy arrays / tensors, ensure native floats
    try:
        import numpy as np
        if isinstance(vec, np.ndarray):
            vec = vec.tolist()
    except Exception:
        pass
    return [float(x) for x in vec]

def bulk_upsert(client: OpenSearch, index: str, ids: List[str], vectors: List[List[float]], texts: List[str]):
    actions = []
    for i, vid in enumerate(ids):
        actions.append({
            "_op_type": "index",
            "_index": index,
            "_id": str(vid),
            "_source": {
                "id": str(vid),
                "embedding": to_float_list(vectors[i]),
                "text": str(texts[i]),
            }
        })
    helpers.bulk(client, actions)
    client.indices.refresh(index=index)

def knn_search(client: OpenSearch, index: str, query_vec: List[float], k: int = 3, source_fields=("id","text")) -> List[Dict]:
    body = {
        "size": int(k),
        "query": {
            "knn": {
                # OpenSearch-native syntax: field name is the key; object has "vector" and "k"
                "embedding": {"vector": to_float_list(query_vec), "k": int(k)}
            }
        },
        "_source": list(source_fields)
    }
    resp = client.search(index=index, body=body)
    return resp.get("hits", {}).get("hits", [])

def main():
    # 1) Load data
    df = pd.read_csv(raw_csv_path)
    if ID_COL not in df.columns or TEXT_COL not in df.columns:
        raise ValueError(f"CSV must have columns '{ID_COL}' and '{TEXT_COL}'. Found: {list(df.columns)}")

    # 2) Compute embeddings with Termite (SciNCL by default)
    t = Termite(kg_credentials=None, verbose=True, model_name=MODEL_NAME)

    # emb_map: row_index -> vector; make row-ordered list aligned with df
    emb_map = t.compute_embeddings(df, model_name=MODEL_NAME)
    embeddings = [emb_map[i] for i in df.index.tolist()]
    dim = len(embeddings[0])

    # 3) Connect to OpenSearch and prepare index
    client = get_client()
    ensure_index(client, INDEX_NAME, dim, recreate=RECREATE_INDEX)

    # 4) Prepare ids/texts and upsert with payloads
    ids   = df[ID_COL].astype(str).tolist()
    texts = df[TEXT_COL].fillna("").astype(str).tolist()
    bulk_upsert(client, INDEX_NAME, ids, embeddings, texts)

    # 5) Query with the first vector
    hits = knn_search(client, INDEX_NAME, embeddings[0], k=TOP_K, source_fields=("id","text"))

    print("\nTop-{} hits:".format(TOP_K))
    for h in hits:
        _id    = h.get("_id")
        score  = float(h.get("_score", 0.0))
        src    = h.get("_source", {})
        text   = src.get("text")
        print(f"  score={score:.4f}  id={_id}  text={text[:120]}")

    # 6) (Optional) Inspect stored fields once
    sample = client.search(index=INDEX_NAME, body={"size":1,"query":{"match_all":{}}})
    keys = list(sample["hits"]["hits"][0]["_source"].keys())
    print("\nStored _source keys on a sample doc:", keys)

if __name__ == "__main__":
    main()



Top-3 hits:
  score=1.0000  id=7fac733d-83ec-4b15-a48d-16a893a2373a  text=Zero-day vulnerabilities pose a significant threat to cybersecurity systems. The kernel trick in SVMs enables efficient 
  score=0.9441  id=5bf26de5-b5da-499c-bcb2-d70ae542792f  text=Zero-day vulnerabilities pose a significant threat to cybersecurity systems. Graph neural networks excel at processing s
  score=0.9423  id=83913121-bc9c-4b12-ac93-5563ebd881a4  text=Reinforcement learning enables agents to learn optimal policies through trial and error. Highly specific datasets of sci

Stored _source keys on a sample doc: ['id', 'embedding', 'text']



Query: What problem in real-world malware labeling does the HNMFk Classifier aim to solve?
Top-5 hits:
  score=0.8113  id=fafed8b2-197d-45f6-ad53-881821264fa4  text=Recurrent neural networks are widely used for sequential data such as speech and text. Adversarial machine learning explores methods to enhance model robustness against attacks. Training deep learning models requires high computational power and GPUs. We propose an efficient distributed out-of-memory implementation of the non-negative matrix factorization (NMF) algorithm for heterogeneous high-performance-computing systems. The proposed implementation is based on prior work on NMFk, which can perform automatic model selection and extract latent variables and patterns from data. In this work, we extend NMFk by adding support for dense and sparse matrix operation on multi-node, multi-GPU systems. The resulting algorithm is optimized for out-of-memory problems where the memory required to factorize a given matrix is greater t

In [ ]:
# test_termite_e2e.py
import os, pandas as pd
from TELF.applications import Termite

# make sure Termite picks OpenSearch
os.environ["EMBEDDING_STORE"] = "opensearch"
os.environ["OS_HOST"] = os.getenv("OS_HOST", "localhost")
os.environ["OS_PORT"] = os.getenv("OS_PORT", "9200")
os.environ["OS_USE_SSL"] = "false"


df = pd.read_csv(raw_csv_path)

t = Termite(kg_credentials=None, verbose=True, model_name="malteos/scincl")

# compute embeddings (dict: row_index -> vector) and align with df order
emb_map = t.compute_embeddings(df, model_name="malteos/scincl")
embeddings = [emb_map[i] for i in df.index]
dim = len(embeddings[0])

index_name = "termite_vectors_test_e2e"

# (re)create index with correct dim
t.store.ensure_index(index=index_name, dim=dim, metric="cosine")

# build payloads so 'text' is present at top level
ids = df["eid"].astype(str).tolist()
payloads = [{"text": txt} for txt in df["abstract"].astype(str).tolist()]

# upsert (ids + vectors + payloads)
t.store.upsert(index_name, ids, embeddings, payloads=payloads)

# --- add near the top ---
MODEL_NAME = "malteos/scincl"   # keep consistent with your index

def embed_text_with_termite(termite, text: str):
    import pandas as pd
    # Use the same column name your DF used (abstract)
    qdf = pd.DataFrame({"abstract": [text]})
    qmap = termite.compute_embeddings(qdf, model_name=MODEL_NAME)
    return qmap[qdf.index[0]]

# Custom query text
query_text = "What problem in real-world malware labeling does the HNMFk Classifier aim to solve?"   # << your text here

# 1) embed the query text
qvec = embed_text_with_termite(t, query_text)

# 2) run kNN against your index
hits = t.store.search(index_name, qvec, k=5, source_fields="id,text")

print("\nQuery:", query_text)
print("Top-5 hits:")
for _id, score, src in hits:
    print(f"  score={score:.4f}  id={_id}  text={src.get('text')}")




Top-3 hits:
  score=1.0000  id=7fac733d-83ec-4b15-a48d-16a893a2373a  text=Zero-day vulnerabilities pose a significant threat to cybersecurity systems. The kernel trick in SVMs enables efficient classification in non-linearly separable data. Graph neural networks excel at processing structured graph data for various applications. Support vector machines are effective in high-dimensional spaces for classification problems. Supervisory Control and Data Acquisition (SCADA) systems often serve as the nervous system for substations within power grids. These systems facilitate real-time monitoring, data acquisition, control of equipment, and ensure smooth and efficient operation of the substation and its connected devices. As the dependence on these SCADA systems grows, so does the risk of potential malicious intrusions that could lead to significant outages or even permanent damage to the grid. Previous work has shown that dimensionality reduction-based approaches, such as Principal Compone